In [1]:
# importing basic libraries to handle data and files
import numpy as np
import pickle

# importing keras components to build and train the model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# for preparing text input for predictions
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
# loading the processed datasets shared by Isha
X_train = np.load("X_train.npy")
X_val = np.load("X_val.npy")
X_test = np.load("X_test.npy")

y_train = np.load("y_train.npy")
y_val = np.load("y_val.npy")
y_test = np.load("y_test.npy")

# checking the shape of each dataset to confirm everything loaded correctly
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (1334, 200)
X_val: (286, 200)
X_test: (287, 200)
y_train: (1334,)
y_val: (286,)
y_test: (287,)


In [4]:
# loading the tokenizer that was already fitted during preprocessing
with open("tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

# checking that it loaded correctly
print(type(tokenizer))

<class 'keras.src.legacy.preprocessing.text.Tokenizer'>


In [5]:
# getting the vocabulary size from the tokenizer
vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary size:", vocab_size)

Vocabulary size: 19251


In [6]:
# building the baseline LSTM model using the vocabulary size from Isha's tokenizer
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64),
    LSTM(64),
    Dense(1, activation="sigmoid")
])

In [7]:
# compiling the model for binary sentiment classification
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [8]:
# training the baseline model with the processed training and validation data
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=32
)

Epoch 1/5
42/42 ━━━━━━━━━━━━━━━━━━━━ 8s 117ms/step - accuracy: 0.4970 - loss: 0.6953 - val_accuracy: 0.4965 - val_loss: 0.6924
Epoch 2/5
42/42 ━━━━━━━━━━━━━━━━━━━━ 4s 105ms/step - accuracy: 0.5165 - loss: 0.6913 - val_accuracy: 0.5385 - val_loss: 0.6907
Epoch 3/5
42/42 ━━━━━━━━━━━━━━━━━━━━ 6s 136ms/step - accuracy: 0.5622 - loss: 0.6800 - val_accuracy: 0.5105 - val_loss: 0.6877
Epoch 4/5
42/42 ━━━━━━━━━━━━━━━━━━━━ 9s 113ms/step - accuracy: 0.6199 - loss: 0.6220 - val_accuracy: 0.5105 - val_loss: 0.6837
Epoch 5/5
42/42 ━━━━━━━━━━━━━━━━━━━━ 5s 118ms/step - accuracy: 0.6072 - loss: 0.5818 - val_accuracy: 0.5315 - val_loss: 0.6896


In [10]:
# evaluating the model on unseen test data to check final performance
test_loss, test_accuracy = model.evaluate(X_test, y_test)

print("Test accuracy:", test_accuracy)

9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.5017 - loss: 0.7069
Test accuracy: 0.5017421841621399


In [13]:
# testing the trained model with a custom input sentence
sample_text = ["this product is amazing and I love it"]

# converting text to sequence using the same tokenizer
sample_seq = tokenizer.texts_to_sequences(sample_text)

# padding to match the input shape used during training
sample_pad = pad_sequences(sample_seq, maxlen=200, padding="post", truncating="post")

# getting prediction
prediction = model.predict(sample_pad)[0][0]

print("Prediction score:", prediction)

# interpreting the result
if prediction >= 0.5:
    print("Positive review")
else:
    print("Negative review")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Prediction score: 0.54885244
Positive review


In [ ]:
# saving the trained model so it can be reused later in the web app V1 (BaseLineModel)
model.save("model.h5")

In [16]:
# 29 Abril 2026
# creating a second version of the model with dropout to reduce overfitting
from tensorflow.keras.layers import Dropout

model_v2 = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64),
    LSTM(64),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

# compiling the improved model
model_v2.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [18]:
# training the improved model with more epochs to see if performance improves
history_v2 = model_v2.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

Epoch 1/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 6s 117ms/step - accuracy: 0.5052 - loss: 0.6936 - val_accuracy: 0.5385 - val_loss: 0.6921
Epoch 2/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 6s 143ms/step - accuracy: 0.5120 - loss: 0.6921 - val_accuracy: 0.5385 - val_loss: 0.6907
Epoch 3/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 9s 117ms/step - accuracy: 0.5397 - loss: 0.6927 - val_accuracy: 0.5524 - val_loss: 0.6901
Epoch 4/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 6s 128ms/step - accuracy: 0.5817 - loss: 0.6803 - val_accuracy: 0.5559 - val_loss: 0.6859
Epoch 5/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 5s 108ms/step - accuracy: 0.5772 - loss: 0.6641 - val_accuracy: 0.5280 - val_loss: 0.6828
Epoch 6/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 6s 140ms/step - accuracy: 0.5855 - loss: 0.6452 - val_accuracy: 0.5524 - val_loss: 0.6769
Epoch 7/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 5s 107ms/step - accuracy: 0.6064 - loss: 0.5977 - val_accuracy: 0.5245 - val_loss: 0.7011
Epoch 8/10
42/42 ━━━━━━━━━━━━━━━━━━━━ 4s 106ms/step - accuracy: 0.6394 - loss: 0.5563 - val_accuracy: 0.

In [19]:
# creating another model version using Bidirectional LSTM to better capture text context
from tensorflow.keras.layers import Bidirectional

model_v3 = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64),
    Bidirectional(LSTM(64)),
    Dense(1, activation="sigmoid")
])

# compiling the new model for binary sentiment classification
model_v3.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [20]:
# training the Bidirectional LSTM model to compare it with the baseline
history_v3 = model_v3.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=32
)

Epoch 1/5
42/42 ━━━━━━━━━━━━━━━━━━━━ 13s 210ms/step - accuracy: 0.5187 - loss: 0.6907 - val_accuracy: 0.5734 - val_loss: 0.6834
Epoch 2/5
42/42 ━━━━━━━━━━━━━━━━━━━━ 9s 215ms/step - accuracy: 0.6717 - loss: 0.6104 - val_accuracy: 0.6713 - val_loss: 0.6158
Epoch 3/5
42/42 ━━━━━━━━━━━━━━━━━━━━ 9s 203ms/step - accuracy: 0.8561 - loss: 0.4126 - val_accuracy: 0.6923 - val_loss: 0.5985
Epoch 4/5
42/42 ━━━━━━━━━━━━━━━━━━━━ 8s 197ms/step - accuracy: 0.9363 - loss: 0.1871 - val_accuracy: 0.7238 - val_loss: 0.5976
Epoch 5/5
42/42 ━━━━━━━━━━━━━━━━━━━━ 11s 216ms/step - accuracy: 0.9783 - loss: 0.0879 - val_accuracy: 0.7343 - val_loss: 0.6279


In [21]:
# evaluating the improved model on the test dataset
test_loss_v3, test_accuracy_v3 = model_v3.evaluate(X_test, y_test)

print("Test accuracy (model_v3):", test_accuracy_v3)

9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 81ms/step - accuracy: 0.7456 - loss: 0.6037
Test accuracy (model_v3): 0.7456446290016174


In [22]:
# testing the improved model with a few custom reviews
test_sentences = [
    "this product is amazing and I love it",
    "this product is terrible and I hate it",
    "the quality is excellent and delivery was fast",
    "very disappointing product and poor service",
    "it is okay, not great but not terrible"
]

# converting the custom text inputs into sequences
test_sequences = tokenizer.texts_to_sequences(test_sentences)

# padding the sequences to match the model input length
test_padded = pad_sequences(test_sequences, maxlen=200, padding="post", truncating="post")

# getting predictions from the improved model
predictions = model_v3.predict(test_padded)

# showing the results in a readable way
for sentence, prediction in zip(test_sentences, predictions):
    score = prediction[0]
    label = "Positive" if score >= 0.5 else "Negative"
    print(f"Text: {sentence}")
    print(f"Score: {score:.4f}")
    print(f"Prediction: {label}")
    print("-" * 50)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 358ms/step
Text: this product is amazing and I love it
Score: 0.8299
Prediction: Positive
--------------------------------------------------
Text: this product is terrible and I hate it
Score: 0.0803
Prediction: Negative
--------------------------------------------------
Text: the quality is excellent and delivery was fast
Score: 0.7960
Prediction: Positive
--------------------------------------------------
Text: very disappointing product and poor service
Score: 0.1760
Prediction: Negative
--------------------------------------------------
Text: it is okay, not great but not terrible
Score: 0.0414
Prediction: Negative
--------------------------------------------------


In [25]:
# saving the best performing model (Bidirectional LSTM)
model_v3.save("model.h5")